# 3. Building Polymers with mBuild

Polymer builders usually give you the right bonding graph and a terrible
configuration. A 50-mer built monomer by monomer comes out as a straight rod.
mBuild splits the problem in two.

1. Decide the **shape** first, as a coarse grained `Path` of beads
2. **Backmap** the path to atoms, so the chemistry follows the shape you chose
3. **Relax** the result, because backmapping strains bonds at the fragment joints

### What you will do here
- Build paths, including branched backbones, branched rings, and composites
- Backmap them to real polymers with CGsmiles
- Minimize with capped displacement, FIRE, and a short NVT run
- Put paths together with the rest of mBuild, solvating, confining between
  surfaces, and wrapping a chain around a ligand

In [32]:
import numpy as np
from collections import Counter
import random

import mbuild as mb
import gmso
from gmso import ForceField
from gmso.parameterization import apply

from mbuild.path import (
    Path, straight_line, cyclic, lamellar, helix, zigzag, knot,
    hard_sphere_random_walk,
)
from mbuild.path.constraints import CuboidConstraint, SphereConstraint
from mbuild.path.termination import (
    Termination, NumSites, NumAttempts, WithinCoordinate,
)
from mbuild.path.bias import TargetCoordinate, AvoidCoordinate
from mbuild.exceptions import PathConvergenceError
from mbuild.simulation import HoomdSimulation, ForcesHandler

import logging
from mbuild import mBuildLogger
mBuildLogger().library_logger.setLevel(logging.ERROR)
gmso.gmso_logger.library_logger.setLevel(logging.ERROR)

oplsaa = ForceField("oplsaa")

## The path module

A `Path` is coordinates plus a bond graph. Nothing chemical about it. The
builders in `mbuild.path` are generators that populate one.

Pass the same `Path` object to several builders and they accumulate, which is
how composite structures get made.

In [33]:
spacing = 0.32

line = straight_line(N=25, spacing=spacing)
line.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [34]:
ring = cyclic(N=30, spacing=spacing)
ring.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [35]:
sheet = lamellar(
    spacing=spacing,
    num_layers=5,
    layer_separation=1.2,
    layer_length=4.0,
)
sheet.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [36]:
walk = hard_sphere_random_walk(
    radius=spacing, bond_length=spacing, termination=40, seed=11
)
walk.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Branching

`hard_sphere_random_walk` can grow a branch off an existing path when you give it

- `initial_point=<int>`, the index of the site to start from
- `connectivity="link-linear"`, which bonds the new segment back to that site
- `path=Path()`, the previously created path to branch from. 

The walk sees every coordinate already in the path, so branches avoid the
backbone and each other.

Name the beads as you go. `bead_name` sets the fragment name that backmapping
looks up later, so backbone and branch can become different chemistries.

### A branched backbone

In [37]:
backbone = hard_sphere_random_walk(
    radius=spacing,
    bond_length=spacing,
    rw_angles=[2.6, 3.14],
    termination=24,
    bead_name="BB"
)

for site in (4, 9, 14, 19):
    hard_sphere_random_walk(
        path=backbone,
        initial_point=site,
        connectivity="link-linear",
        bead_name="BR",
        radius=spacing,
        bond_length=spacing,
        termination=10,
        seed=site,
    )

degrees = dict(backbone.bond_graph.degree()).values()
print(len(backbone.coordinates), "beads |", backbone.bond_graph.number_of_edges(),
      "bonds | max degree", max(degrees))
backbone.visualize(radius=spacing)

64 beads | 63 bonds | max degree 3


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### A branched ring

Same move on a cyclic path. `cyclic` closes the ring for you, then branches grow
outward from evenly spaced sites.

This architecture is worth pausing on, because it is the one reused at the end of
the notebook. A ring has no chain ends, so every bead has at least two neighbors,
and the branch points have three. That mix of degrees is what makes it a
genuinely branched macromolecule rather than a long chain, and it is worth
knowing how backmapping copes with it. A CGsmiles fragment carries a bonding
descriptor for every bond the bead could make, and a descriptor with nothing to
connect to is capped with a hydrogen. So one fragment covers the branch points
and the plain ring beads alike, and the branch points just come out with one
fewer hydrogen. You do not need a separate fragment per degree.

Where the branches point matters as much as how many there are. Grown without a
bias they splay outward and you get a star. Biased inward they fold over the
middle of the ring and you get a cavity, which is the host architecture built in
the encapsulation example below.

In [38]:
ring = cyclic(N=24, spacing=spacing+0.12, bead_name="RG")

for site in range(0, 24, 4):
    hard_sphere_random_walk(
        path=ring,
        initial_point=site,
        connectivity="link-linear",
        bead_name="BR",
        radius=spacing,
        bond_length=spacing,
        termination=12,
        seed=site + 1,
    )

print(len(ring.coordinates), "beads |", ring.bond_graph.number_of_edges(), "bonds")
ring.visualize(radius=spacing)

96 beads | 96 bonds


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Composite conformations

Chain random walk builders on an existing `Path` and you get a molecule that is ordered in some
places and disordered in others. Semicrystalline polymers look like this. Chains
fold into lamellar crystallites, and the segments that leave one crystallite and
run into another are the amorphous tie molecules that hold the solid together.

Build it in that order. Place the two lamellar blocks first, since their geometry
is the part you actually care about, then let a random walk find its own way from
one to the other. Two arguments do that work.

- `bias=TargetCoordinate(...)` sorts each set of trial steps so the ones heading
  toward the target get tried first
- `termination=WithinCoordinate(...)` ends the walk once a step lands within a
  cutoff of the target

Those two are different kinds of thing. `WithinCoordinate` is a target condition,
so the walk counts as a success when it arrives. `NumAttempts` is a safeguard.
Every target condition has to be met for a walk to succeed, while any safeguard
triggering stops the walk without one.

The tie walk uses a slightly smaller `radius` than the blocks do, because it has
to approach a bead that its own hard sphere test is telling it to avoid. Leave
the radius at the bond length and the walk can never get close enough to arrive.

In [39]:
sheet = lamellar(initial_point=(0,0,0), layer_length=3, layer_separation=0.7, spacing=spacing, num_layers=6, bead_name="PE")
sheet.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The sheet above is one folded block. Its last bead is a chain end sitting at the
edge of that block, so it is the natural place to keep growing.

Passing `path=sheet` hands the walk the coordinates that are already there, and
`initial_point=len(sheet.coordinates)-1` starts it from that terminal bead.
`connectivity="link-linear"` is what makes this one molecule rather than two,
since it bonds the first new bead back to the bead it started from. Without it
the walk drops an unbonded segment into the same `Path`.

The walk keeps `bead_name="PE"`, the same name the sheet uses. Both regions are
the same chemistry here and only the conformation differs, so one CGsmiles
fragment backmaps the whole thing later. Names are set per bead though, so a
different name is all it would take to make the disordered segment a different
chemistry.

In [40]:
# Run a Random Walk from the terminal end of the sheet path:
hard_sphere_random_walk(
    path=sheet,
    initial_point=len(sheet.coordinates)-1,
    bead_name="PE",
    termination=60,
    radius=spacing,
    bond_length=spacing,
    seed=12,
    connectivity="link-linear",
)
sheet.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

That gives a crystallite with one disordered tail. Running the same walk from
`initial_point=0` grows a second tail off the other chain end, which is the
shape a real semicrystalline chain has when it leaves a lamella at both ends.

The call is identical apart from the starting index.

In [41]:
# Run a Random Walk from the other end of the sheet path:
hard_sphere_random_walk(
    path=sheet,
    initial_point=0,
    bead_name="PE",
    termination=60,
    radius=spacing,
    bond_length=spacing,
    seed=12,
    connectivity="link-linear",
)
sheet.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The entire workflow to create this polymer chain conformation is shown below, and involves one function call, and a simple for loop:

In [42]:
# Create the lamellar sheet first
sheet = lamellar(initial_point=(0,0,0), layer_length=3, layer_separation=0.7, spacing=spacing, num_layers=6, bead_name="PE")

# We have 2 starting points to run a bonded random walk from:
for starting_index in [0, len(sheet.coordinates) - 1]:
    hard_sphere_random_walk(
        path=sheet,
        initial_point=starting_index,
        bead_name="PE",
        termination=60,
        radius=spacing,
        bond_length=spacing,
        seed=12,
        connectivity="link-linear",
    )

sheet.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [43]:
### Using plugins to form a tie-chain:

In [44]:
sheetA = lamellar(initial_point=(0,0,0), layer_length=3, layer_separation=0.7, spacing=spacing, num_layers=6, bead_name="PE", left_to_right=False)
sheetB = lamellar(initial_point=(4,4,2), layer_length=3, layer_separation=0.7, spacing=spacing, num_layers=6, bead_name="PE")
sheet_system = sheetA + sheetB
sheet_system.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [45]:
# Run a Random Walk from the terminal end of this path:
target_site = TargetCoordinate(target_coordinate=sheet_system.coordinates[0], weight=0.25)
termination = WithinCoordinate(target_coordinate=sheet_system.coordinates[0], distance=spacing, tolerance=0.1)

hard_sphere_random_walk(
    path=sheet_system,
    bead_name="PE",
    initial_point=len(sheet_system.coordinates)-1,
    bias=target_site,
    termination=termination,
    radius=spacing,
    bond_length=spacing,
    seed=10,
    connectivity="link-linear",
)
sheet_system.add_edge(0, len(sheet_system.coordinates)-1)

sheet_system.visualize(radius=spacing)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The tie-chain walk arrived, so the resulting path is one continuous, linear chain traversing ordered regions (lamellar paths) and an unordered region (random walk).

Worth knowing that `WithinCoordinate` is a target condition and reaching it is
what makes the walk a success. Pair it with a `NumAttempts` safeguard inside a
`Termination` and a walk that cannot converge stops on its own, though it stops
without raising, so check `Termination.success` before adding the closing edge.

---

# CGsmiles backmapping

`Path.backmap` replaces every bead with a molecular fragment and keeps the
bead's position, so the atomistic chain inherits the shape you designed.

Fragments are written as SMILES with bonding descriptors.

- `{#PEO=[>]COC[<]}` says the bead named `PEO` is `-CH2-O-CH2-`, with a head and
  a tail
- `[>]` pairs with `[<]`, and `[$]` pairs with itself
- A bead needs at least as many descriptors as its degree in the bond graph

For branch points the labels matter, not just the count. CGsmiles fills bonds
using the first compatible descriptor in atom order, so if every descriptor on a
branching bead carries the same label you get a comb where you wanted a branched
chain. Give each bond type its own directed pair, like `[>bb]` and `[<bb]` for
the backbone and `[<br]` for the branch.

In [46]:
fragments = "{#BB=[>bb]C([<br])C[<bb],#BR=[>br]CC[<br]}"

branched_pe = backbone.backmap(fragments)
branched_pe.name = "branched_pe"
print(branched_pe.n_particles, "atoms |", branched_pe.n_bonds, "bonds")
branched_pe.visualize()

386 atoms | 385 bonds


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The result keeps the coarse grained hierarchy. Every child of the compound is
one bead, holding the atoms that bead resolved to, so you can still reason about
the structure in terms of the path you designed.

In [47]:
for child in list(branched_pe.children)[:5]:
    print(child.name, child.n_particles, "atoms")

for child in list(branched_pe.children)[-5:]:
    print(child.name, child.n_particles, "atoms")

BB 7 atoms
BB 6 atoms
BB 6 atoms
BB 6 atoms
BB 5 atoms
BR 6 atoms
BR 6 atoms
BR 6 atoms
BR 6 atoms
BR 7 atoms


Backmap the branched ring the same way. Here the backbone is
poly(ethylene oxide) and the branches are polyethylene.

In [48]:
ring_polymer = ring.backmap("{#RG=[>bb]C([<br])OC[<bb],#BR=[>br]CC[<br]}")
ring_polymer.name = "ring_polymer"
print(ring_polymer.n_particles, "atoms")
ring_polymer.visualize()

600 atoms


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

And the sheet. Every bead on this path is named `PE`, so one fragment covers the
whole thing and the lamellar block and the tie chain come out as the same
chemistry. The architecture is what distinguishes them here, not the monomer.

In [49]:
semi_crystalline = sheet_system.backmap("{#PE=[>]CC[<]}")
semi_crystalline.name = "semi_crystalline"
print(semi_crystalline.n_particles, "atoms |", semi_crystalline.n_bonds, "bonds")
semi_crystalline.visualize()

1178 atoms | 1177 bonds


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

# Energy minimization in mBuild

Backmapped structures have strained bonds at the fragment joints. Straight into
NVT and they explode. The order that works is

1. **Capped displacement**, which caps how far any particle moves per step, so
   the worst overlaps relax without anything flying apart
2. **FIRE**, a proper minimizer, once the structure is no longer pathological
3. **A short NVT run**, only if you want to sample away from the built shape

`HoomdSimulation` takes a Compound or a Path plus a GMSO ForceField, builds the
HOOMD state and forces, and writes relaxed coordinates back onto the compound in
place.

In [27]:
chain = semi_crystalline
sim = HoomdSimulation(chain, forcefield=oplsaa, r_cut=1.2)
print(chain.n_particles, "atoms staged in HOOMD")

1388 atoms staged in HOOMD


### Capped displacement

A large `dt` is right here. It lets `max_displacement` dominate the position
update, which turns the step into a capped steepest descent. Too small a `dt`
and the cap never engages, leaving you with plain NVE on a stiff potential.

In [28]:
sim.cap_displacement(n_steps=2000, dt=1, max_displacement=1e-3)
energies = sim.get_energy()["potential_energy"]
print("start:", energies[0], " after capped displacement:", energies[-1])

*Warning*: charge.pppm: RMS error of 0.556623 is probably too high! 0.556623 0.556623


start: 1471915.743872502  after capped displacement: 9796.257457344354


### FIRE

In [29]:
sim.fire(n_steps=3000, n_iterations=5)
print("after FIRE:", sim.get_energy()["potential_energy"][-1])
chain.visualize()

*Warning*: charge.pppm: RMS error of 0.556623 is probably too high! 0.556623 0.556623
IOStream.flush timed out


after FIRE: 8298.88550250543


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Overlay the original path on the relaxed chain. The conformation you designed
survived the round trip, which is the whole point of building this way.

In [30]:
overlay = mb.clone(chain)
overlay.add(sheet_system.to_compound())
overlay.visualize(bead_size=0.5)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

This system of a single linear chain with ordered regions and non-ordered regions in a vacuum might not be a target system to study, but it serves to illustrates how one could design an interative, and programmatically controllable, process towards building a more dense semi-crystalline system

In [ ]:
layer_separation = 0.7
sheet = lamellar(initial_point=(0,0,0), layer_length=3, layer_separation=layer_separation, spacing=spacing, num_layers=6, bead_name="PE")
sheet_compound = sheet.to_compound()
sheet_box = mb.fill_box(compound=sheet_compound, n_compounds=5, box=[10,10,10], overlap=layer_separation*3)

sheet_system = Path.from_compound(compound=sheet_box)
sheet_system.visualize(radius=spacing)

In [ ]:
terminal_indices = [i for i, p in enumerate(sheet_box.particles()) if p.n_direct_bonds == 1]
random.shuffle(terminal_indices) # Shuffle the list to randomize the order
random_pairs = [(terminal_indices[i], terminal_indices[i+1]) for i in range(0, len(terminal_indices), 2)]

tie_chain_probability = 0.8
for pair in random_pairs:
    random_num = np.random.random()
    make_tie_chain = random_num <= tie_chain_probability

    if make_tie_chain:
        bias = TargetCoordinate(sheet_system.coordinates[pair[1]], weight=0.25)
        termination = WithinCoordinate(sheet_system.coordinates[pair[1]], distance=spacing, tolerance=0.1)
    else:
        bias = None
        termination = NumSites(30)

    hard_sphere_random_walk(
        path=sheet_system,
        radius=spacing,
        bead_name="PE",
        bond_length=spacing,
        bias=bias,
        termination=termination,
        initial_point=pair[0],
        connectivity="link-linear"
    )

    if make_tie_chain:
        sheet_system.add_edge(len(sheet_system.coordinates) - 1, pair[1])


sheet_system.visualize(radius=spacing)

It is worth wrapping the relaxation steps into a single function, since you will run it on everything
below.

In [ ]:
aa_sheet = sheet_system.backmap("{}")

In [ ]:
comp = sheet_system.to_compound()
comp.save("/Users/cj4006/multi-sheet.mol2")

In [ ]:
aa_sheet = comp.ba

In [ ]:
def relax(
    compound,
    forcefield=None,
    cap_steps=2000,
    fire_steps=2000,
    fire_iterations=5,
    r_cut=1.2
):
    sim = HoomdSimulation(compound, forcefield=forcefield, r_cut=r_cut)
    sim.cap_displacement(n_steps=cap_steps, dt=1, max_displacement=1e-3)
    sim.fire(n_steps=fire_steps, n_iterations=fire_iterations)
    return sim

---

# Putting it together with the rest of mBuild

A backmapped `Path` is an ordinary mBuild `Compound`, so packing, cloning, geometric transformations, and all other mBuild Compound functionality can be used.

## Solvating a chain

A random walk gives a compact coil, which packs into a box much better than a
long branched chain does. This saves simulation time required to relax an extended chain in a solvent and perform volume annealing from low denisty to higher target densities.

In [ ]:
coil = hard_sphere_random_walk(
    radius=0.45, bond_length=0.45, termination=30, seed=8, bead_name="PE"
)
solute = coil.backmap("{#PE=[>]COC[<]}")
solute.name = "polymer"
relax(solute)
solute.visualize()

In [ ]:
hexane = mb.load("CCCCCC", smiles=True)
hexane.name = "hexane"

solution = mb.fill_box(
    compound=[solute, hexane],
    n_compounds=[2, 350],
    box=[5, 5, 5],
    seed=3,
    overlap=0.12,
)
print(solution.n_particles, "atoms")
solution.visualize()

## Chains confined between two surfaces

Build one surface, clone and flip it, and stack them with a gap. Then run random
walks in the gap.

Two arguments do the confinement work.

- `volume_constraint` rejects any step that leaves the region
- `include_compound` makes the walk see the surface atoms, so chains do not grow
  through the walls

Then backmap the walks and add them to the slab, so the same construction ends up
as a fully atomistic confined film.

In [ ]:
from mbuild.lib.surfaces import Betacristobalite

bottom = Betacristobalite()
bottom.translate_to((0, 0, 0))
Lx, Ly, Lz = bottom.get_boundingbox().lengths

gap = 4.0
top_surface = mb.clone(bottom)
top_surface.rotate(np.pi, around=(1, 0, 0))
top_surface.translate_to((0, 0, gap))

slab = mb.Compound(subcompounds=[bottom, top_surface])
print(slab.n_particles, "surface atoms | gap", gap, "nm")
slab.visualize()

In [ ]:
bead_radius = 0.30
free_height = gap - Lz - 2 * bead_radius

region = CuboidConstraint(
    Lx=Lx - 2 * bead_radius,
    Ly=Ly - 2 * bead_radius,
    Lz=free_height,
    center=(0, 0, gap / 2),
    pbc=(True, True, False)
)

confined = Path()
for i in range(25):
    occupied = slab.xyz
    if len(confined.coordinates):
        occupied = np.vstack([occupied, confined.coordinates])
    starts = region.find_low_density_points(
        n_candidates=200, points=occupied, buffer=bead_radius
    )
    for start in starts:
        try:
            hard_sphere_random_walk(
                path=confined,
                initial_point=start,
                bead_name="PEG",
                radius=bead_radius,
                bond_length=bead_radius + 0.01,
                volume_constraint=region,
                include_compound=slab,
                termination=Termination([NumSites(25), NumAttempts(2000)]),
                seed=i,
            )
            break
        except PathConvergenceError:
            continue

print(len(confined.coordinates), "beads in the gap")

In [ ]:
interface = mb.clone(slab)
interface.add(confined.to_compound())
interface.visualize(bead_size=0.5, periodic_bond_opacity=0.3)

The walks are still just a path (i.e., coarse-grain representation), so backmap them on their own and then add the
atoms to the slab. Here they become poly(ethylene glycol).

Nothing in the backmapping knows the surfaces are there. The surfaces did their
work earlier, when `include_compound` stopped the walks from growing into them,
and the shape they produced is carried through to the atoms.

In [ ]:
peg = confined.backmap("{#PEG=[>]COC[<]}")
peg.name = "peg"
print(peg.n_particles, "atoms in", len(list(peg.children)), "chains")

film = mb.clone(slab)
film.add(peg)
print(film.n_particles, "atoms total")
film.visualize(periodic_bond_opacity=0.25)

## Building a branched polymer host for small-molecule encapsulation

Put a small molecule at the origin, build a ring path around it, then grow
branches biased toward the center so they close over the guest.

The reason this example is here is not the application. It is the separation
between geometry and chemistry, which is the thing the path module buys you.

First you say what shape you want.

> A cyclic, branched architecture with a cavity holding this molecule.

Then, separately, you say what it is made of.

> Now give that architecture the chemistry of polymer X.

Those are two independent decisions, and the same path can be backmapped to
different chemistries without rebuilding anything. The usual alternative is to
construct a chemically specific polymer by hand and hope it adopts the
morphology you wanted. Here the workflow runs architecture, then chemistry, then
a simulation ready system.

- `include_compound=ligand` keeps the walk from growing through the molecule
- `TargetCoordinate` biases each trial step toward a point, with `weight`
  between 0 and 1 setting how strongly

In [ ]:
ligand = mb.load("c1ccc2c(c1)sc1ccccc12", smiles=True)   # dibenzothiophene
ligand.name = "ligand"
ligand.translate_to((0, 0, 0))
print(ligand.n_particles, "atoms")
ligand.visualize()

### The architecture

Two bead names, `RING` and `ARM`. The ring beads that carry a branch have degree
three and the rest have degree two, but both get backmapped with the same
fragment. A bonding descriptor with nothing to connect to is capped with a
hydrogen, so one fragment covers both cases. This is the same trick the branched
ring earlier in the notebook uses.

In [ ]:
cage_spacing = 0.375
branch_sites = range(0, 18, 2)

cage = cyclic(N=18, spacing=cage_spacing, bead_name="RING")
cage.coordinates -= cage.coordinates.mean(axis=0)   # center the ring on the ligand

for site in branch_sites:
    hard_sphere_random_walk(
        path=cage,
        initial_point=site,
        connectivity="link-linear",
        bead_name="ARM",
        radius=cage_spacing,
        bond_length=cage_spacing,
        termination=15,
        seed=site,
        include_compound=ligand,
        bias=TargetCoordinate((0, 0, 0), weight=0.35),
    )

degrees = Counter(d for _, d in cage.bond_graph.degree())
print(len(cage.coordinates), "beads |", dict(Counter(cage.beads.tolist())))
print("degrees:", dict(degrees), "| the 3s are the junctions, the 1s are arm tips")
cage.visualize(radius=cage_spacing)

### Now the chemistry

The path above is pure geometry. Backmapping is where it becomes a molecule, and
nothing about the shape is decided here.

The `RING` fragment carries three descriptors, two along the ring and one for a
branch. Ring beads without a branch leave the third unused and it becomes a
hydrogen, so branch points come out as CH and the rest as CH2 with no special
casing. Build the same path twice with different fragments and you get two
different polymers with the same architecture and the same cavity.

Relaxation here passes `forcefield=None`, which falls back to UFF. The
`oplsaa.xml` we have been using raises a type ambiguity on the fused ring carbons
of dibenzothiophene, and UFF is generated on the fly for whatever you hand it.
For pushing a backmapped structure off its strained starting geometry that is
plenty. You would still type with a real force field before production.

In [ ]:
CHEMISTRIES = {
    "polyethylene": "{#RING=[>bb]C([<br])C[<bb],#ARM=[>br]CC[<br]}",
    "poly(ethylene oxide)": "{#RING=[>bb]C([<br])OC[<bb],#ARM=[>br]COC[<br]}",
}

complexes = {}
for name, fragments in CHEMISTRIES.items():
    host = cage.backmap(fragments)
    host.name = "host"
    host_guest = mb.Compound(name="host_guest")
    host_guest.add(mb.clone(ligand))
    host_guest.add(host)
    #relax(host_guest, forcefield=None)   # UFF, see below
    complexes[name] = host_guest
    print(f"{name:22s} {host.n_particles:4d} atoms in the host | complex box "
          f"{np.round(host_guest.get_boundingbox().lengths, 2)} nm")

In [ ]:
complexes["poly(ethylene oxide)"].visualize()

Two chemistries, one geometric construction, and the cavity survives both. Swap
the fragment strings for a chemistry you actually care about and the rest of the
notebook is unchanged.

### Putting it in water

The complex is an ordinary Compound, so `mb.solvate` works on it exactly as it
would on a single molecule. This is also the setup that would actually test the
encapsulation. A hydrophobic host holding a hydrophobic guest in water is the
situation where the hydrophobic effect would keep the arms closed, rather than
the bias that put them there.

Since PACKMOL fills anything it can reach, counting how far out water stays away
from the guest says something about whether the arms really closed. The sulfur is
a convenient marker, since it is the only one in the system, and with the
polyethylene host every oxygen present belongs to a water.

In [ ]:
water = mb.load("O", smiles=True)
water.name = "water"

solvated = mb.solvate(
    solute=complexes["polyethylene"],
    solvent=water,
    n_solvent=2400,
    box=[5.0, 5.0, 5.0],
    overlap=0.16,
    seed=7,
)
print(solvated.n_particles, "atoms |", dict(Counter(c.name for c in solvated.children)))

sulfur = [p for p in solvated.particles() if p.element.symbol == "S"][0]
oxygens = np.array([p.pos for p in solvated.particles() if p.element.symbol == "O"])
distances = np.linalg.norm(oxygens - sulfur.pos, axis=1)

print("\n r (nm)   waters   bulk would be")
for r in (0.7, 1.1, 1.5, 1.8):
    print(f"{r:6.1f}   {int((distances < r).sum()):6d}   {(4 / 3) * np.pi * r**3 * 33.4:11.0f}")

solvated.visualize()

### Why build it this way

This is to demonstrate a programmatic initialization workflow. The arms closed because the walk was biased inward, not
because the chemistry made them close, so what you have is a starting
configuration rather than a result.

The point is that the configuration comes from what you already expect the system
to look like, written down as arguments. Cavity size, arm length, number of arms,
guest chemistry, and polymer chemistry. Each one is a parameter, so the same script rebuilds the system
exactly, extends to a different guest or a different polymer without a rewrite,
and lets you sweep whichever variable you care about.

<h1 style="color: green;">Exercise</h1>

Design a star polymer.

1. Make a short `cyclic` or `straight_line` path as the core, named `"CORE"`
2. Grow 5 or 6 arms off it with `hard_sphere_random_walk`, named `"ARM"`
3. Backmap it, using a core fragment with enough descriptors for the branch
   degree, and an arm fragment of a different chemistry
4. Relax it with the `relax` helper and visualize
5. **BONUS:** Trying using a bias (`TargetCoordinate`) to control the conformation of the branching arms. For example: make them extend outward from the the ring, or collapse onto the ring.
6. **BONUS 2:** Solvate this structure with a solvent of your choice (water, hexane, ethanol, or some combination). Make sure your box is large enough to fit the star polymer compound. Hint: use the `star.get_boundingbox()` function, which also accepts a box-length padding value.

Tip. Start from the branched backbone cell and change what you branch off.

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>

Run the cell below to see one solution.

In [ ]:
core = cyclic(N=6, spacing=0.45, bead_name="CORE")

for site in range(6):
    hard_sphere_random_walk(
        path=core, initial_point=site, connectivity="link-linear",
        bead_name="ARM", radius=0.45, bond_length=0.45,
        termination=10, seed=site + 20,
    )

star = core.backmap("{#CORE=[>bb]C([<br])C[<bb],#ARM=[>br]COC[<br]}")
star.name = "star"
print(star.n_particles, "atoms")
relax(star)
star.visualize()

In [ ]:
water = mb.load("O", smiles=True)
hexane = mb.load("CCCCCC", smiles=True)
system = mb.fill_box(
    compound=[star, hexane],
    n_compounds=[1, 300],
    box=star.get_boundingbox(pad_box=0.5)
)

system.visualize()

---

### Recap

- A `Path` is shape without chemistry, and shape is the part that is hard to get right
- Branches come from `initial_point` plus `connectivity="link-linear"`
- Builders accumulate on one `Path`, and `add_edge` joins the segments
- `backmap` turns beads into fragments and keeps the conformation
- Capped displacement then FIRE, and NVT only when you want to leave the shape behind
- Paths compose with packing, surfaces, and any other Compound

Next up, running all of this as a parameter study with signac.

In [63]:
mon = mb.load("C1C=CC2C1C3CC2C=C3", smiles=True)
liquid_mon_sphere = mb.fill_sphere(compound=mon, n_compounds=225, sphere=[5, 5, 5, 2.5])
liquid_mon_sphere.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [68]:
from mbuild.path.constraints import SphereConstraint

In [98]:
vol = SphereConstraint(center=(0,0,0), radius=10)

path = Path()

for i in range(200):
    if i == 0:
        points = liquid_mon_sphere.xyz
    else:
        points = np.concat([path.coordinates, liquid_mon_sphere.xyz])
    
    starts = vol.sample_candidates(points=points, n_candidates=1000, buffer=0.4)
    for start in starts:
        try:
            hard_sphere_random_walk(
                include_compound=liquid_mon_sphere,
                path=path,
                volume_constraint=vol,
                initial_point=start,
                radius=0.35,
                bond_length=0.351,
                termination=4
            )
            break
        except PathConvergenceError:
            pass
        

In [99]:
path.visualize(radius=0.35)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [80]:
liquid_mon_sphere.xyz

array([[5.2670944, 5.6137105, 5.0949929],
       [5.3553092, 5.525569 , 5.0132515],
       [5.4583977, 5.5903626, 4.961038 ],
       ...,
       [4.4052418, 5.0068182, 6.4533062],
       [4.2195857, 4.82168  , 6.4172917],
       [4.3153274, 4.5923167, 6.494214 ]], shape=(4950, 3))

In [51]:
mon.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.